In [2]:
import os
import time
import pickle
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
from sklearn.model_selection import cross_val_score

warnings.filterwarnings('ignore')

print("=========================================")
print("  E-COMMERCE MODEL TRAINING & EVALUATION")
print("=========================================\n")

# Ensure output directories exist
os.makedirs('../../backend/models', exist_ok=True)
os.makedirs('../../backend/artifacts', exist_ok=True)

# 1. Load the safely scaled train/test splits
train_df = pd.read_csv('../../data/processed/train.csv')
test_df = pd.read_csv('../../data/processed/test.csv')

X_train = train_df.drop('Revenue', axis=1)
y_train = train_df['Revenue']

X_test = test_df.drop('Revenue', axis=1)
y_test = test_df['Revenue']

print(f"Training Data: {X_train.shape[0]} rows")
print(f"Testing Data: {X_test.shape[0]} rows\n")

# 2. Dynamic Imbalance Calculation for XGBoost
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
dynamic_scale_pos_weight = neg_count / pos_count
print(f"Calculated scale_pos_weight for XGBoost: {dynamic_scale_pos_weight:.2f}\n")

# 3. Initialize Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=10, class_weight='balanced', random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=6, scale_pos_weight=dynamic_scale_pos_weight, random_state=42, eval_metric='logloss')
}

results = []

# Prepare ROC figure
plt.figure(figsize=(10, 8))

print("Training models and generating artifacts...\n")

for name, model in models.items():
    print(f"--> Training {name}...")
    
    # Train with time tracking
    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time
    
    # Save the individual model
    filename = name.lower().replace(" ", "_") + ".pkl"
    with open(f"../../backend/models/{filename}", "wb") as f:
        pickle.dump(model, f)
    
    # Predict
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # Calculate Core Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)
    
    results.append({
        "Model": name,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "ROC-AUC": round(roc_auc, 4),
        "Training Time (s)": round(training_time, 2)
    })
    
    # Generate & Save Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Bounce", "Purchase"])
    disp.plot(cmap='Blues', values_format='d')
    plt.title(f"{name} Confusion Matrix")
    plt.savefig(f"../../backend/artifacts/{filename.replace('.pkl', '_cm.png')}")
    plt.close()
    
    # Accumulate ROC Curve Data
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {roc_auc:.3f})")

# Finalize and Save ROC Curve Comparison
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend(loc="lower right")
plt.savefig('../../backend/artifacts/roc_comparison.png')
plt.close()

# 4. Process and Save Metrics
# Sorted by ROC-AUC as the primary metric for imbalanced data
results_df = pd.DataFrame(results).sort_values(by="ROC-AUC", ascending=False).reset_index(drop=True)
results_df.to_csv('../../backend/artifacts/model_metrics.csv', index=False)

print("\n--- Model Comparison Metrics ---")
print(results_df.to_markdown())

# 5. Dynamically Determine and Save the Best Model
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
print(f"\nWinner determined by ROC-AUC: {best_model_name}")

with open('../../backend/models/best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print("Saved best_model.pkl for standard API consumption.")

# 6. Robustness Check: Cross-Validation on the Winner
print(f"\nPerforming 5-Fold Cross Validation on {best_model_name}...")
cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring="roc_auc")
print(f"CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# 7. Feature Importance (Assuming Tree-Based Winner like XGBoost/RF)
if hasattr(best_model, 'feature_importances_'):
    print("\nGenerating Feature Importance artifacts...")
    importance = pd.DataFrame({
        "Feature": X_train.columns,
        "Importance": best_model.feature_importances_
    }).sort_values(by="Importance", ascending=False)
    
    # Save CSV for Next.js Tables
    importance.to_csv('../../backend/artifacts/feature_importance.csv', index=False)
    
    # Save Plot
    plt.figure(figsize=(10, 8))
    importance.head(20).plot(x="Feature", y="Importance", kind="barh", color='teal', legend=False)
    plt.title(f"Top 20 Feature Importances ({best_model_name})")
    plt.xlabel("Relative Importance")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('../../backend/artifacts/feature_importance.png')
    plt.close()

print("\nPipeline execution complete. Ready for SHAP analysis and API integration!")

  E-COMMERCE MODEL TRAINING & EVALUATION

Training Data: 9864 rows
Testing Data: 2466 rows

Calculated scale_pos_weight for XGBoost: 5.46

Training models and generating artifacts...

--> Training Logistic Regression...
--> Training Decision Tree...
--> Training Random Forest...
--> Training XGBoost...

--- Model Comparison Metrics ---
|    | Model               |   Accuracy |   Precision |   Recall |   F1-Score |   ROC-AUC |   Training Time (s) |
|---:|:--------------------|-----------:|------------:|---------:|-----------:|----------:|--------------------:|
|  0 | XGBoost             |     0.8796 |      0.5906 |   0.7251 |     0.651  |    0.9168 |                0.13 |
|  1 | Logistic Regression |     0.8629 |      0.5396 |   0.7853 |     0.6397 |    0.9163 |                0.07 |
|  2 | Random Forest       |     0.8982 |      0.7569 |   0.5052 |     0.606  |    0.9139 |                0.79 |
|  3 | Decision Tree       |     0.8244 |      0.4612 |   0.7932 |     0.5833 |    0.8547 | 

<Figure size 1000x800 with 0 Axes>